# D1 local ablation (qwen3.5:4b via Ollama) on Colab

Runs `scripts/local_ablation.py` — the V1 local tier arm `D1_local_single` — against Ollama's `qwen3.5:4b` inside a Colab runtime instead of your laptop.

**Before running:** Runtime → Change runtime type → pick a GPU (T4 is fine, free tier). CPU-only will also work, just slower.

Only the `D1` arm is implemented locally today (single local agent). The hosted `A_deterministic` / `C_crew_llm` arms are a separate, not-yet-built piece of work — out of scope here.

Order: (optional) mount the Drive cache → install Ollama → restore or pull the model → get the repo onto the runtime → restore corpora and link rows to Drive → install Python deps → fetch whatever the cache lacked → build the dev manifest → `freeze` → `run` → `summarise` → save to Drive.

**Second and later sessions** with the Drive cache: the model pull, both corpus fetches and every already-completed row are skipped, so a session goes from mount to `run` in a few minutes instead of the better part of an hour.

In [ ]:
!apt-get update && apt-get install -y zstd

## 0. (optional) Google Drive cache

A fresh Colab runtime spends most of its setup time downloading: the 3.4 GB model, the flaws.cloud CloudTrail corpus (~250 MB), the DEDALE Winlogbeat day (~1.4 GB). None of it changes between runs. This cell mounts Drive and names a cache folder; the cells below **restore** from it when it holds something and **fetch** only when it does not, and the save cell in section 6 fills it. Rows are written straight into Drive, so a run interrupted by a runtime reset resumes where it stopped (`run` skips rows already on disk).

Set `USE_DRIVE = False` to run exactly as before, with nothing persisted.

What is and is not cached, and why: the model blobs and the corpora are cached because they are identical bytes every time. The manifest, the freeze and the summary are **not** cached: they are rebuilt on this runtime every session, because the telemetry digest depends on the runtime (see the caveat in section 5) and a freeze belongs to the commit that ran.

In [ ]:
USE_DRIVE = True

import os, shutil, subprocess
from pathlib import Path

DRIVE_ROOT = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/ath_colab_cache')
    for sub in ('ollama_models', 'data_external', 'rows'):
        (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)
    print('Drive cache at', DRIVE_ROOT)
    for sub in ('ollama_models', 'data_external', 'rows'):
        size = subprocess.run(['du', '-sh', str(DRIVE_ROOT / sub)], capture_output=True, text=True).stdout.split()[0]
        print(f'  {sub:15s} {size}')
else:
    print('Drive cache disabled; everything is fetched fresh and nothing persists')


def cached(sub: str, marker: str = '') -> bool:
    """Whether the Drive cache holds ``sub`` (and, when given, ``marker`` inside it)."""
    if DRIVE_ROOT is None:
        return False
    target = DRIVE_ROOT / sub / marker if marker else DRIVE_ROOT / sub
    return target.exists() and any(target.iterdir()) if target.is_dir() else target.exists()


def restore(sub: str, dest: Path) -> bool:
    """Copy the cached ``sub`` into ``dest`` (local disk, fast to read). False when empty."""
    if not cached(sub):
        return False
    dest.mkdir(parents=True, exist_ok=True)
    subprocess.run(['rsync', '-a', '--info=progress2', str(DRIVE_ROOT / sub) + '/', str(dest) + '/'], check=True)
    return True


def save(sub: str, source: Path) -> None:
    """Copy ``source`` into the Drive cache under ``sub``; existing files are not re-copied."""
    if DRIVE_ROOT is None or not source.exists():
        return
    subprocess.run(['rsync', '-a', '--ignore-existing', '--info=progress2', str(source) + '/', str(DRIVE_ROOT / sub) + '/'], check=True)
    print('saved', source, '->', DRIVE_ROOT / sub)


## 1. Install and start Ollama

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Section 0 defines the Drive helpers. If it was skipped, run without a cache rather than fail.
if 'DRIVE_ROOT' not in globals():
    print('NOTE: section 0 (Drive cache) was not run; continuing with no cache')
    from pathlib import Path
    import os, shutil, subprocess
    DRIVE_ROOT = None
    def cached(sub, marker=''): return False
    def restore(sub, dest): return False
    def save(sub, source): return None

# Restore the Ollama model store from Drive BEFORE the daemon starts, so `ollama pull`
# below finds the blobs and returns in seconds. The store is copied to local disk rather
# than pointed at Drive: a 3.4 GB gguf memory-mapped over the Drive FUSE mount loads slowly.
OLLAMA_MODELS = Path('/root/.ollama/models')
if restore('ollama_models', OLLAMA_MODELS):
    print('restored Ollama model store from Drive')
else:
    print('no cached model store; the pull below downloads it')


In [ ]:
import subprocess, time, urllib.request

def daemon_up() -> bool:
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/version', timeout=2)
        return True
    except Exception:
        return False

if daemon_up():
    print('ollama daemon already up')
else:
    # Detached, logging to a file: a daemon whose stdout/stderr go to an unread PIPE
    # blocks once the buffer fills, and one tied to this cell dies with the kernel.
    log = open('/content/ollama.log', 'ab')
    subprocess.Popen(['ollama', 'serve'], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    for attempt in range(120):
        if daemon_up():
            print(f'ollama daemon is up (after {attempt + 1} s)')
            break
        time.sleep(1)
    else:
        raise SystemExit('ollama daemon did not start; see /content/ollama.log')

# Re-run this cell after any runtime reset or reconnect: freeze/run refuse without the daemon.
!ollama ps


In [ ]:
# Section 0 defines the Drive helpers. If it was skipped, run without a cache rather than fail.
if 'DRIVE_ROOT' not in globals():
    print('NOTE: section 0 (Drive cache) was not run; continuing with no cache')
    from pathlib import Path
    import os, shutil, subprocess
    DRIVE_ROOT = None
    def cached(sub, marker=''): return False
    def restore(sub, dest): return False
    def save(sub, source): return None

# Fast when the store was restored (verifies the manifest, downloads nothing); a full
# download otherwise. Either way the digest the freeze records is the same weights.
!ollama pull qwen3.5:4b
save('ollama_models', Path('/root/.ollama/models'))


## 2. Get the repo onto the runtime

The repo is a **private** GitHub repo (`<private-owner>/agentic-threat-hunter`), so a plain `git clone` will 404. Two options — use whichever is easier:

**Option A — clone with a token.** Create a fine-grained GitHub PAT with read access to this one repo, add it as a Colab secret named `GH_TOKEN` (key icon in the left sidebar), then run the cell below.

**Option B — upload a zip.** Zip your local working copy (skip `.git`, `data/external/`, `reports/local/`) and upload/mount it, then skip the clone cell and just `%cd` into the extracted folder.

In [ ]:
# Option A: clone with a token stored as a Colab secret
from google.colab import userdata

GH_TOKEN = userdata.get("GH_TOKEN")
REPO = "<private-owner>/agentic-threat-hunter"
BRANCH = "m14-real-data-validation"

!apt-get -qq install -y git-lfs && git lfs install
!git clone -b {BRANCH} https://{GH_TOKEN}@github.com/{REPO}.git /content/agentic-threat-hunter
!cd /content/agentic-threat-hunter && git lfs pull

In [ ]:
%cd /content/agentic-threat-hunter

### Restore corpora and rows from Drive

`data/external/` is copied down from the cache when it holds anything (the two fetch cells below then skip themselves). `reports/local/dev/rows/` becomes a **symlink into Drive**, so every row is persisted the moment it is written and a rerun after a reset resumes. Rows from a different manifest hash or prompt version live in their own subdirectory and are never mixed, so the cache may safely hold rows from several sessions.

In [ ]:
# Section 0 defines the Drive helpers. If it was skipped, run without a cache rather than fail.
if 'DRIVE_ROOT' not in globals():
    print('NOTE: section 0 (Drive cache) was not run; continuing with no cache')
    from pathlib import Path
    import os, shutil, subprocess
    DRIVE_ROOT = None
    def cached(sub, marker=''): return False
    def restore(sub, dest): return False
    def save(sub, source): return None

REPO = Path('/content/agentic-threat-hunter')

if restore('data_external', REPO / 'data' / 'external'):
    print('restored data/external from Drive')

rows_dir = REPO / 'reports' / 'local' / 'dev' / 'rows'
if DRIVE_ROOT is not None:
    if rows_dir.is_symlink():
        rows_dir.unlink()
    elif rows_dir.exists():
        # Anything the clone brought along (normally nothing: rows are gitignored) moves
        # into the cache first, so no row is shadowed by the symlink.
        save('rows', rows_dir)
        shutil.rmtree(rows_dir)
    rows_dir.parent.mkdir(parents=True, exist_ok=True)
    rows_dir.symlink_to(DRIVE_ROOT / 'rows', target_is_directory=True)
    print('rows ->', os.readlink(rows_dir))
    existing = sorted(p.relative_to(DRIVE_ROOT / 'rows') for p in (DRIVE_ROOT / 'rows').rglob('*.json'))
    print(f'{len(existing)} row file(s) already in the cache')
    for rel in existing[:10]:
        print('  ', rel)


## 3. Python deps

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .

## 4. Fetch external corpora + build the dev manifest

`data/external/` is gitignored (only the manifest index is committed) so it has to be fetched fresh on a new runtime.

The dev build only needs `flaws_cloud` (for the `flaws_cloud` dev bundle) plus the DEDALE D02 Winlogbeat hour (for the ten injected dev cases) — fetch those directly rather than `fetch_external.py`'s full manifest, since that also includes `k8s_ci`, whose pinned GCS log artifact has since expired (test-infra garbage-collects old CI logs) and would abort the whole fetch with a 404.

In [ ]:
flaws = Path('data/external/flaws_cloud')
if flaws.exists() and any(flaws.iterdir()):
    print('flaws_cloud present (restored from Drive or fetched earlier); skipping fetch')
else:
    subprocess.run(['python', 'scripts/fetch_external.py', 'flaws_cloud'], check=True)


In [ ]:
# Section 0 defines the Drive helpers. If it was skipped, run without a cache rather than fail.
if 'DRIVE_ROOT' not in globals():
    print('NOTE: section 0 (Drive cache) was not run; continuing with no cache')
    from pathlib import Path
    import os, shutil, subprocess
    DRIVE_ROOT = None
    def cached(sub, marker=''): return False
    def restore(sub, dest): return False
    def save(sub, source): return None

winlogbeat = Path('data/external/dedale/winlogbeat')
if winlogbeat.exists() and any(winlogbeat.iterdir()):
    print('DEDALE Winlogbeat present (restored from Drive or fetched earlier); skipping fetch')
else:
    subprocess.run(['python', 'scripts/dedale_fetch_hours.py', '--days', '2'], check=True)
save('data_external', Path('data/external'))


In [ ]:
!python scripts/local_inject_dedale.py

In [ ]:
!python scripts/local_manifest.py build

## 5. freeze / smoke / run / summarise (D1 v3, the bounded investigator)

`freeze` pins the model digest, daemon version, the investigator's prompt hashes and bounds against the manifest hash; `run` refuses if any of them drifted. Rows are written under `reports/local/dev/rows/D1_qwen3.5-4b/m<manifest12>_d1-investigator-v3/` and the row key carries the manifest hash, so rows from another manifest or another prompt version are never mixed. `run` is resumable: rerunning after a disconnect skips rows already on disk.

**Manifest hash caveat.** The telemetry digest depends on the runtime (numpy/pandas/Python): the laptop builds `2298b0e8`, this Colab runtime builds `e4115893`, for byte-identical cases. Build, freeze, run and summarise must all happen on this runtime, which the cells above and below do.

**Smoke first.** Five representative cases (V1 obvious malicious + lineage discovery, V2 subtle malicious, V7 and V8 benign look-alikes, flaws CASE-071 ambiguous). Read the per-case investigation table before launching the full twenty: tool choices should differ when gaps differ, benign cases may stay benign, hypotheses must not copy the detector, new evidence ids should appear, and no call should hit the 768-token cap. If that fails, stop and change the investigator (a new prompt version = a new commit + re-freeze), do not tune on the full set.

**What this run measures.** v3 (`59fadc1`) is the second of at most three tuning passes and has not been measured yet; running it as-is costs no pass. Read `LINK-2 recovered / defined` from the investigation table: the harness recovers it 9/9 under the offline oracle replay (`python scripts/local_replay.py --path link`, no model), so a zero here belongs to the model. Each round now records the raw model reply and the prompt's sha256, so a downloaded row can be replayed offline and its prompts matched byte for byte.

In [ ]:
!python scripts/local_ablation.py freeze --model qwen3.5:4b --arm D1

In [ ]:
!python scripts/local_ablation.py run --model qwen3.5:4b --arm D1 --repeat 1 --seed 0 --only dedale_injected_dev:V1/CASE-001 dedale_injected_dev:V2/CASE-001 dedale_injected_dev:V7/CASE-001 dedale_injected_dev:V8/CASE-001 flaws_cloud/CASE-071

In [ ]:
!python scripts/local_ablation.py summarise --model qwen3.5:4b --arm D1 --repeat 1 > /dev/null
text = open('reports/local/dev/SUMMARY_D1_qwen3.5-4b_rep1.md').read()
print(text[text.index('## Per case: investigation'):])

In [ ]:
# Every model claim of the smoke rows, to check they are not detector paraphrases
import glob, json
for path in sorted(glob.glob('reports/local/dev/rows/D1_qwen3.5-4b/*/*.json')):
    row = json.load(open(path))['row']; inv = row['state'].get('investigation', {})
    print('=' * 90); print(row['corpus'], row['case_id'], '| label', row.get('labels', {}).get('verdict', 'unlabelled'), '| disposition', inv.get('final_disposition'), '| probes', inv.get('probes_run'), '| truncated', inv.get('output_truncated'))
    print('  gap:', inv.get('evidence_gap')); print('  reason:', inv.get('tool_choice_reason'))
    for c in row['state']['claims']:
        if c['source'] == 'llm': print('  ', c['type'], c['statement'][:200], c['evidence_ids'])
    print('  rejected:', [r['reason'][:70] for r in row['state']['rejected_claims']], '| links:', row.get('label_scores', {}).get('links'))
    for r in inv.get('rounds', []):
        print('  round', r.get('round'), '| chose', (r.get('chosen_probe') or {}).get('tool'), '| new ids', r.get('new_evidence_ids_returned'), '| prompt', str(r.get('prompt_sha256'))[:12], r.get('prompt_chars'), 'chars')
        print('    raw:', (r.get('raw') or '')[:400])

### Full run (only after the smoke table is sane)

The five smoke rows are ordinary rows under the same identity and are skipped, so the full run adds the remaining fifteen.

In [ ]:
!python scripts/local_ablation.py run --model qwen3.5:4b --arm D1 --repeat 1 --seed 0

In [ ]:
!python scripts/local_ablation.py summarise --model qwen3.5:4b --arm D1 --repeat 1

In [ ]:
print(open("reports/local/dev/SUMMARY_D1_qwen3.5-4b_rep1.md").read())

## 6. Persist to Drive and (optional) pull results back down

With the Drive cache on, the rows are already in Drive (the `rows/` symlink) and the model store and corpora were saved when they were fetched. The cell below re-saves them all once more so nothing depends on the earlier cells having run, then zips `reports/local/dev/` for download. The zip is still the artifact to bring to the laptop: it also holds the manifest, the freeze and the summary of *this* runtime, which the cache deliberately does not keep.

In [ ]:
# Section 0 defines the Drive helpers. If it was skipped, run without a cache rather than fail.
if 'DRIVE_ROOT' not in globals():
    print('NOTE: section 0 (Drive cache) was not run; continuing with no cache')
    from pathlib import Path
    import os, shutil, subprocess
    DRIVE_ROOT = None
    def cached(sub, marker=''): return False
    def restore(sub, dest): return False
    def save(sub, source): return None

save('ollama_models', Path('/root/.ollama/models'))
save('data_external', Path('data/external'))
if DRIVE_ROOT is not None and not Path('reports/local/dev/rows').is_symlink():
    save('rows', Path('reports/local/dev/rows'))
# A copy of the manifest, freeze and summary beside the rows, named by manifest hash, so a
# later session can tell which rows belong to which runtime without the zip.
if DRIVE_ROOT is not None:
    import json as _json
    digest = _json.load(open('reports/local/dev/MANIFEST.json'))['manifest_hash'][:12]
    side = DRIVE_ROOT / 'runs' / digest
    side.mkdir(parents=True, exist_ok=True)
    for name in ('MANIFEST.json', 'MANIFEST.md', 'ENVIRONMENT_qwen3.5-4b.json', 'SUMMARY_D1_qwen3.5-4b_rep1.json', 'SUMMARY_D1_qwen3.5-4b_rep1.md'):
        src_path = Path('reports/local/dev') / name
        if src_path.exists():
            shutil.copy2(src_path, side / name)
    print('manifest/freeze/summary copied to', side)


In [ ]:
# zip follows the rows/ symlink into Drive by default (it would store the link only with -y).
!zip -r -q /content/dev_results.zip reports/local/dev
from google.colab import files
files.download('/content/dev_results.zip')
